Get Gold that is over $3,000 an ounnce and place in seperate Delta Table

In [1]:
from spark_session import get_spark
import pyspark.sql 
from pyspark.sql import functions as F
spark = get_spark('test')
x = spark.sql("select * from gld")
x = x.withColumn('Yr', F.year(F.col("Date_")))
df_high = x.filter(F.col("Gld_Close")>3000)
df_high.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gld_highs")
df_high.show()


+----+----------+---------+----+
| _c0|     Date_|Gld_Close|  Yr|
+----+----------+---------+----+
|3826|2025-03-18|   3035.1|2025|
|3827|2025-03-19|   3035.9|2025|
|3828|2025-03-20|   3040.0|2025|
|3829|2025-03-21|   3018.2|2025|
|3830|2025-03-24|   3013.1|2025|
|3831|2025-03-25|   3023.7|2025|
|3832|2025-03-26|   3020.9|2025|
|3833|2025-03-27|   3060.2|2025|
|3834|2025-03-28|   3086.5|2025|
|3835|2025-03-31|   3122.8|2025|
|3836|2025-04-01|   3118.9|2025|
|3837|2025-04-02|   3139.9|2025|
|3838|2025-04-03|   3097.0|2025|
|3839|2025-04-04|   3012.0|2025|
|3842|2025-04-09|   3056.5|2025|
|3843|2025-04-10|   3155.2|2025|
|3844|2025-04-11|   3222.2|2025|
|3845|2025-04-14|   3204.8|2025|
|3846|2025-04-15|   3218.7|2025|
|3847|2025-04-16|   3326.6|2025|
+----+----------+---------+----+
only showing top 20 rows


Run Yfinance Pipeline and store in Delta Table Format Locally. 

In [2]:
import yfinance as yf
import numpy as np 
import pandas as pd 
import datetime
from spark_session import get_spark
from pyspark.sql import functions as F
from delta.tables import DeltaTable
TICKERS = {
    'GC=F':  'Gold',
    'SI=F':  'Silver',
    'PL=F':  'Platinum',
    'PA=F':  'Palladium',
    'HG=F':  'Copper',
    'CL=F':  'WTI Crude Oil',
    'BZ=F':  'Brent Crude',
    'XEL':   'Xcel Energy',
    'CVX':   'Chevron',
    'BAC':    'Bank of America',
    'BAH':     'Booze Allen Hamilton'
}
spark = get_spark("stocks")
#grab yfinance data 
ticker_lst =['PL=F', 'GC=F','SI=F','HG=F','PA=F', 'CL=F', 'BZ=F', 'XEL', 'CVX', 'BAC', 'BAH']
dt = yf.download(ticker_lst, start='2020-01-01', group_by='ticker')
#Download historical data for the last year
dt = pd.DataFrame(data=dt)
dt_f = dt.reset_index()
dt_f.columns = ['_'.join(col).strip() for col in dt_f.columns.values] #transform white space to underscore.
dt_f.columns = ["".join(col).replace('=','_') for col in dt_f.columns.values] #change '=' to underscore.
dt_f = np.round(dt_f, decimals=2)
        #print(dt_f.columns)

df_spark = spark.createDataFrame(dt_f)
df_spark = df_spark.withColumn('DateKey', F.date_format(F.col('Date_'),'yyyyMMdd').cast("int"))
df_spark = df_spark.withColumn('Date_',F.date_format(F.col('Date_'),'yyyy-MM-dd'))
df_spark.printSchema()
df_spark.show()
#load into delta table
df_spark.write.mode("overwrite").format("delta").option("inferSchema","true").saveAsTable("stocks_2")


C:\Users\test\AppData\Local\Temp/ipykernel_22720/1511851613.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dt = yf.download(ticker_lst, start='2020-01-01', group_by='ticker')
[*********************100%***********************]  11 of 11 completed


root
 |-- Date_: string (nullable = true)
 |-- BZ_F_Open: double (nullable = true)
 |-- BZ_F_High: double (nullable = true)
 |-- BZ_F_Low: double (nullable = true)
 |-- BZ_F_Close: double (nullable = true)
 |-- BZ_F_Volume: long (nullable = true)
 |-- PL_F_Open: double (nullable = true)
 |-- PL_F_High: double (nullable = true)
 |-- PL_F_Low: double (nullable = true)
 |-- PL_F_Close: double (nullable = true)
 |-- PL_F_Volume: double (nullable = true)
 |-- PA_F_Open: double (nullable = true)
 |-- PA_F_High: double (nullable = true)
 |-- PA_F_Low: double (nullable = true)
 |-- PA_F_Close: double (nullable = true)
 |-- PA_F_Volume: double (nullable = true)
 |-- GC_F_Open: double (nullable = true)
 |-- GC_F_High: double (nullable = true)
 |-- GC_F_Low: double (nullable = true)
 |-- GC_F_Close: double (nullable = true)
 |-- GC_F_Volume: double (nullable = true)
 |-- XEL_Open: double (nullable = true)
 |-- XEL_High: double (nullable = true)
 |-- XEL_Low: double (nullable = true)
 |-- XEL_Clos

Profile the Delta Table schema, datatypes, and history. 

In [3]:
from spark_session import get_spark
import pyspark.sql
import pyspark.sql.functions as F
from delta.tables import DeltaTable

spark = get_spark('read_tbls')
tbl = DeltaTable.forName(spark, 'stocks')
#get the history of the table
tbl.history().show()
#check schema of delta table
tbl.toDF().printSchema()
#optimize with Zorder on datekey columns
#tbl.optimize().executeZOrderBy("DateKey")
tbl.toDF().show()


+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2026-08-02 01:26:...|  NULL|    NULL|CREATE OR REPLACE...|{isV1SaveAsTableO...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 8, n...|        NULL|Apache-Spark/4.1....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------

Stop Spark Session in case there are errors. 

In [3]:
from spark_session import get_spark
import pyspark.sql 
from pyspark.sql import functions as F
spark = get_spark('stop')
spark.stop()

In [5]:
from spark_session import get_spark
import pyspark.sql 
from pyspark.sql import functions as F
spark = get_spark('stop')
spark.sql("drop table if exists stocks_2")
spark.stop()